# KFP Pipeline with Model Registry — RHOAI Demo

**~15 minutes** | Build a Kubeflow Pipeline that trains a model and registers it in the RHOAI Model Registry.

> **Story**: *"Define your ML workflow as code, run it on the cluster, and automatically register the resulting model — all from a notebook."*

## What this notebook does
1. Defines a 3-step KFP pipeline: **prepare data** → **train model** → **register in Model Registry**
2. Compiles and submits it to the RHOAI pipeline server
3. Verifies the model appears in the Model Registry

## Setup

In [ ]:
!pip install -q kfp boto3 scikit-learn requests

In [ ]:
import os
from pathlib import Path

# --- Pipeline server ---
PIPELINE_URL = "https://ds-pipeline-dspa-rhoai-playground.apps.ocp.xlwsd.sandbox1213.opentlc.com"
NAMESPACE = "rhoai-playground"

# --- Model Registry ---
# Use internal URL from workbench; external URL if running outside the cluster
MODEL_REGISTRY_URL = "https://demo-registry.rhoai-model-registries.svc:8443"
# MODEL_REGISTRY_URL = "https://demo-registry-rest.apps.ocp.xlwsd.sandbox1213.opentlc.com"  # external
MR_API = f"{MODEL_REGISTRY_URL}/api/model_registry/v1alpha3"

# --- MinIO ---
MINIO_ENDPOINT = "http://minio-service.minio.svc.cluster.local:9000"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "pipelines"

# --- Proxy (set these env vars if behind a corporate proxy) ---
HTTPS_PROXY = os.environ.get("https_proxy", "")
NO_PROXY = os.environ.get("no_proxy", "")

# --- Auth token ---
sa_token_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
if sa_token_path.exists():
    TOKEN = sa_token_path.read_text().strip()
    print("Using service account token")
else:
    TOKEN = os.popen("oc whoami -t").read().strip()
    print(f"Using oc token: {TOKEN[:12]}...")

if HTTPS_PROXY:
    print(f"Proxy configured: https_proxy={HTTPS_PROXY}")
    print(f"  no_proxy={NO_PROXY}")

---
## Define the Pipeline

Three components, each running in its own container on the cluster:

In [ ]:
from kfp import dsl, compiler
from typing import NamedTuple

BASE_IMAGE = "python:3.11-slim"


@dsl.component(base_image=BASE_IMAGE, packages_to_install=["scikit-learn", "boto3", "pandas"])
def prepare_data(
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
) -> str:
    """Generate a synthetic dataset and upload to MinIO."""
    import boto3
    import pandas as pd
    from sklearn.datasets import make_classification
    from io import StringIO

    X, y = make_classification(
        n_samples=500, n_features=10, n_informative=5,
        n_redundant=2, random_state=42,
    )
    df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(10)])
    df["target"] = y

    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    s3 = boto3.client(
        "s3",
        endpoint_url=minio_endpoint,
        aws_access_key_id=minio_access_key,
        aws_secret_access_key=minio_secret_key,
    )
    data_key = "data/training_data.csv"
    s3.put_object(Bucket=minio_bucket, Key=data_key, Body=csv_buffer.getvalue())
    print(f"Uploaded dataset to s3://{minio_bucket}/{data_key} ({len(df)} rows)")
    return data_key


@dsl.component(base_image=BASE_IMAGE, packages_to_install=["scikit-learn", "boto3", "pandas", "joblib"])
def train_model(
    data_key: str,
    minio_endpoint: str,
    minio_access_key: str,
    minio_secret_key: str,
    minio_bucket: str,
) -> NamedTuple("TrainOutputs", [("accuracy", float), ("model_key", str)]):
    """Train a RandomForest and save to MinIO."""
    import boto3
    import pandas as pd
    import joblib
    import io
    from collections import namedtuple
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    s3 = boto3.client(
        "s3",
        endpoint_url=minio_endpoint,
        aws_access_key_id=minio_access_key,
        aws_secret_access_key=minio_secret_key,
    )

    obj = s3.get_object(Bucket=minio_bucket, Key=data_key)
    df = pd.read_csv(io.BytesIO(obj["Body"].read()))

    X = df.drop("target", axis=1)
    y = df["target"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"Model accuracy: {acc:.4f}")

    buf = io.BytesIO()
    joblib.dump(model, buf)
    buf.seek(0)
    model_key = "models/random_forest.pkl"
    s3.put_object(Bucket=minio_bucket, Key=model_key, Body=buf.getvalue())
    print(f"Model saved to s3://{minio_bucket}/{model_key}")

    TrainOutputs = namedtuple("TrainOutputs", ["accuracy", "model_key"])
    return TrainOutputs(acc, model_key)


@dsl.component(base_image=BASE_IMAGE, packages_to_install=["requests"])
def register_model(
    model_key: str,
    accuracy: float,
    registry_url: str,
    minio_bucket: str,
    token: str,
):
    """Register the trained model in the RHOAI Model Registry."""
    import requests
    import json

    api = f"{registry_url}/api/model_registry/v1alpha3"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    # Step 1: Register the model (or find existing)
    resp = requests.post(
        f"{api}/registered_models",
        headers=headers,
        json={"name": "kfp-random-forest", "description": "RandomForest trained via KFP pipeline"},
        verify=False,
    )
    if resp.status_code == 409:
        r = requests.get(f"{api}/registered_models", headers=headers, verify=False)
        model_id = [m for m in r.json()["items"] if m["name"] == "kfp-random-forest"][0]["id"]
        print(f"Model already registered (id={model_id})")
    else:
        resp.raise_for_status()
        model_id = resp.json()["id"]
        print(f"Registered model id={model_id}")

    # Step 2: Create a version
    import time
    version_name = f"v{int(time.time())}-kfp"
    resp = requests.post(
        f"{api}/registered_models/{model_id}/versions",
        headers=headers,
        json={
            "name": version_name,
            "registeredModelId": model_id,
            "description": f"Trained by KFP pipeline, accuracy={accuracy:.4f}",
            "customProperties": {
                "accuracy": {"metadataType": "MetadataDoubleValue", "double_value": accuracy},
                "pipeline": {"metadataType": "MetadataStringValue", "string_value": "kfp-train-register"},
            },
        },
        verify=False,
    )
    resp.raise_for_status()
    version_id = resp.json()["id"]
    print(f"Created version {version_name} id={version_id}")

    # Step 3: Create model artifact
    model_uri = f"s3://{minio_bucket}/{model_key}"
    resp = requests.post(
        f"{api}/model_versions/{version_id}/artifacts",
        headers=headers,
        json={
            "name": f"random-forest-artifact-{version_name}",
            "uri": model_uri,
            "description": "Serialized RandomForestClassifier (joblib)",
            "modelFormatName": "sklearn",
            "modelFormatVersion": "1.5",
            "artifactType": "model-artifact",
        },
        verify=False,
    )
    resp.raise_for_status()
    print(f"Created artifact: {model_uri}")
    print("Model registered in Model Registry!")

In [ ]:
def _set_proxy(task):
    """Inject proxy env vars into a pipeline task if configured."""
    if HTTPS_PROXY:
        task.set_env_variable("https_proxy", HTTPS_PROXY)
        task.set_env_variable("HTTPS_PROXY", HTTPS_PROXY)
    if NO_PROXY:
        task.set_env_variable("no_proxy", NO_PROXY)
        task.set_env_variable("NO_PROXY", NO_PROXY)
    return task


@dsl.pipeline(name="train-and-register", description="Train a model and register it in RHOAI Model Registry")
def train_register_pipeline(
    minio_endpoint: str = MINIO_ENDPOINT,
    minio_access_key: str = MINIO_ACCESS_KEY,
    minio_secret_key: str = MINIO_SECRET_KEY,
    minio_bucket: str = MINIO_BUCKET,
    registry_url: str = MODEL_REGISTRY_URL,
    token: str = TOKEN,
):
    data_task = _set_proxy(prepare_data(
        minio_endpoint=minio_endpoint,
        minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key,
        minio_bucket=minio_bucket,
    ))

    train_task = _set_proxy(train_model(
        data_key=data_task.output,
        minio_endpoint=minio_endpoint,
        minio_access_key=minio_access_key,
        minio_secret_key=minio_secret_key,
        minio_bucket=minio_bucket,
    ))

    _set_proxy(register_model(
        model_key=train_task.outputs["model_key"],
        accuracy=train_task.outputs["accuracy"],
        registry_url=registry_url,
        minio_bucket=minio_bucket,
        token=token,
    ))

---
## Compile the Pipeline

In [ ]:
PIPELINE_FILE = "train_register_pipeline.yaml"
compiler.Compiler().compile(
    pipeline_func=train_register_pipeline,
    package_path=PIPELINE_FILE,
)
print(f"Pipeline compiled to {PIPELINE_FILE}")

---
## Submit the Pipeline Run

In [ ]:
from kfp.client import Client

client = Client(
    host=PIPELINE_URL,
    existing_token=TOKEN,
    namespace=NAMESPACE,
    verify_ssl=False,
)

run = client.create_run_from_pipeline_package(
    PIPELINE_FILE,
    run_name="kfp-train-register-demo",
    arguments={},
)
print(f"Run submitted: {run.run_id}")
print(f"View in dashboard: Develop & train > Pipelines > Runs")

---
## Wait for Completion & Verify

Wait for the pipeline run to finish (typically 2-3 minutes), then verify the model in the registry.

In [ ]:
import time

print("Waiting for pipeline run to complete...")
completed = client.wait_for_run_completion(run_id=run.run_id, timeout=300)
print(f"Run status: {completed.state}")

In [ ]:
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

headers = {"Authorization": f"Bearer {TOKEN}"}

resp = requests.get(f"{MR_API}/registered_models", headers=headers, verify=False)
models = resp.json()
print(f"Models in registry: {models['size']}")
for m in models.get("items", []):
    print(f"  - {m['name']} (id={m['id']})")

    versions = requests.get(
        f"{MR_API}/registered_models/{m['id']}/versions",
        headers=headers, verify=False,
    ).json()
    for v in versions.get("items", []):
        print(f"    Version: {v['name']} — {v.get('description', '')}")

        artifacts = requests.get(
            f"{MR_API}/model_versions/{v['id']}/artifacts",
            headers=headers, verify=False,
        ).json()
        for a in artifacts.get("items", []):
            print(f"      Artifact: {a.get('uri', 'N/A')} ({a.get('modelFormatName', '?')})")

---
## What to Show in the UI

### Pipeline Run
1. **Develop & train > Pipelines > Runs** — select `kfp-train-register-demo`
2. Click the run to see the pipeline graph: `prepare_data` → `train_model` → `register_model`
3. Click each node to see logs, inputs, and outputs

### Model Registry
1. Go to **Model Registry** in the left nav
2. Select the `demo-registry` registry
3. Find `kfp-random-forest` — click to see version `v1-kfp` with accuracy metadata
4. The artifact shows the MinIO URI where the model is stored

### MinIO
- Browse `s3://pipelines/models/random_forest.pkl` to see the serialized model
- Browse `s3://pipelines/data/training_data.csv` to see the generated dataset

> **Talking point**: *"The entire ML workflow — data prep, training, evaluation, and model registration — runs as a reproducible pipeline on the cluster. Every run is versioned, every model is tracked."*